# N02 — Base model (LightGBM teacher) + TreeSHAP ground truth

**Purpose**: Train the black-box teacher. Save TreeSHAP (log-odds scale) as ground-
truth attribution, teacher's prob, score, and internal train/val split indices
(surrogates reuse the same split for fair early stopping).

**Outputs** (`outputs/N02/`):
- `bb_model_{name}.txt` (LGB booster)
- `bb_shap_{name}.npy`, `bb_prob_{name}.npy`, `bb_score_{name}.npy`
- `bb_score_train_{name}.npy`, `bb_score_test_{name}.npy`
- `train_val_idx_{name}.pkl`: positional indices into X_train.

Random state = 42 (model-training seed; separate from data split seed 317).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pickle, json, numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import shap

from decentra._utils import logit, transform_logit_to_score

RS = 42
IN_DIR = Path('../outputs/N01'); OUT = Path('../outputs/N02'); OUT.mkdir(parents=True, exist_ok=True)
with open(IN_DIR/'datasets.pkl','rb') as f: datasets = pickle.load(f)
print('Ready. Datasets:', list(datasets))

## 1. Fit teacher + save TreeSHAP / prob / score

In [ ]:
PARAMS = dict(max_depth=-1, num_leaves=63, n_estimators=1000, learning_rate=0.05,
              subsample=0.8, colsample_bytree=0.8, min_child_samples=50,
              random_state=RS, n_jobs=-1)
meta_all = {}

for name, d in datasets.items():
    X_tr, X_te = d['X_train'], d['X_test']
    y_tr, y_te = d['y_train'], d['y_test']
    # internal train/val for teacher early stopping (also for surrogates)
    tr_pos, val_pos = train_test_split(np.arange(len(X_tr)), test_size=0.2,
                                        stratify=y_tr, random_state=RS)
    X_tr_i, X_val = X_tr.iloc[tr_pos], X_tr.iloc[val_pos]
    y_tr_i, y_val = y_tr.iloc[tr_pos], y_tr.iloc[val_pos]

    clf = lgb.LGBMClassifier(**PARAMS)
    clf.fit(X_tr_i, y_tr_i, eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False)])
    booster = clf.booster_

    prob_te = clf.predict_proba(X_te)[:, 1]
    prob_tr = clf.predict_proba(X_tr)[:, 1]
    shap_vals = shap.TreeExplainer(clf).shap_values(X_te)
    if isinstance(shap_vals, list): shap_vals = shap_vals[1]  # binary
    score_te = transform_logit_to_score(prob_te)
    score_tr = transform_logit_to_score(prob_tr)

    auc_te = roc_auc_score(y_te, prob_te)
    booster.save_model(str(OUT/f'bb_model_{name}.txt'))
    np.save(OUT/f'bb_shap_{name}.npy', shap_vals.astype(np.float32))
    np.save(OUT/f'bb_prob_{name}.npy', prob_te.astype(np.float32))
    np.save(OUT/f'bb_score_{name}.npy', score_te.astype(np.int32))
    np.save(OUT/f'bb_score_train_{name}.npy', score_tr.astype(np.int32))
    np.save(OUT/f'bb_score_test_{name}.npy', score_te.astype(np.int32))
    with open(OUT/f'train_val_idx_{name}.pkl','wb') as f:
        pickle.dump({'train_pos': tr_pos, 'val_pos': val_pos}, f)

    meta_all[name] = {'AUC_test': float(auc_te), 'best_iteration': int(clf.best_iteration_)}
    print(f'{name}: AUC_test={auc_te:.4f}, best_iter={clf.best_iteration_}')

with open(OUT/'meta.json','w') as f: json.dump(meta_all, f, indent=2)